# 🚀 Moebius Inpainting + MiniCPM-V 4.6 Vision GPU Server
This notebook installs dependencies, sets up **hustvl/Moebius** (image inpainting) and **Ollama `minicpm-v4.6`** (`https://ollama.com/library/minicpm-v4.6`, visual understanding), launches the single-file GPU server (`moebius_minicpm_server.py`) with **batch inference + dynamic GPU micro-batching + GPU VRAM/concurrency thresholds**, and exposes a public **Cloudflared** (`trycloudflare.com`) HTTPS URL.

Compatible with:
- **Google Colab CLI** (`colab new --gpu T4` / `colab exec -f devscripts/moebius_minicpm_server.ipynb`)
- **Kaggle CLI** (`kaggle kernels push`)
- **Interactive Jupyter / Colab / Kaggle Web Sessions**


In [ ]:
# @title 1. Check GPU & Runtime Environment
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python     : {sys.version.split()[0]}")
print(f"Platform   : {platform.platform()}")
print(f"Working Dir: {Path.cwd()}")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: nvidia-smi not found. Server will fall back to MPS/CPU if CUDA is unavailable.")


In [ ]:
# @title 2. Install Python Packages, Ollama (`minicpm-v4.6`), Cloudflared & Moebius Weights
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

# 2.1 Install Python packages required by Moebius & FastAPI server
pip_pkgs = [
    "fastapi",
    "uvicorn",
    "pydantic",
    "pillow",
    "numpy",
    "opencv-python-headless",
    "diffusers",
    "transformers",
    "accelerate",
    "huggingface_hub",
    "einops",
    "timm",
    "omegaconf",
    "pyyaml",
    "safetensors",
]
print("📦 Installing Python packages ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pip_pkgs], check=True)

# 2.2 Install Ollama & pull minicpm-v4.6
ollama_model = os.environ.get("OLLAMA_MODEL", "minicpm-v4.6")
if not shutil.which("ollama") and platform.system().lower() == "linux":
    print("🦙 Installing Ollama on Linux ...")
    if not shutil.which("zstd"):
        subprocess.run(["bash", "-c", "apt-get update -qq && apt-get install -y -qq zstd pciutils lshw"], check=False)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

if shutil.which("ollama"):
    env = os.environ.copy()
    env.setdefault("OLLAMA_NUM_PARALLEL", os.environ.get("MAX_VISION_CONCURRENCY", "2"))
    env.setdefault("OLLAMA_MAX_LOADED_MODELS", "1")
    env.setdefault("OLLAMA_FLASH_ATTENTION", "1")
    env.setdefault("OLLAMA_KEEP_ALIVE", "24h")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    print(f"📥 Pulling Ollama model '{ollama_model}' ...")
    subprocess.run(["ollama", "pull", ollama_model], check=False)

# 2.3 Ensure cloudflared binary is available
if not shutil.which("cloudflared") and not Path("./cloudflared").is_file():
    if platform.system().lower() == "linux":
        arch = "arm64" if platform.machine().lower() in ("aarch64", "arm64") else "amd64"
        url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
        print(f"🌐 Downloading cloudflared ({arch}) ...")
        subprocess.run(["curl", "-fsSL", url, "-o", "./cloudflared"], check=True)
        Path("./cloudflared").chmod(0o755)

print("✅ All packages and binaries installed!")


In [ ]:
# @title 3. Locate or Materialize `moebius_minicpm_server.py`
import base64
import zlib
from pathlib import Path

CANDIDATE_PATHS = [
    Path("./devscripts/moebius_minicpm_server.py"),
    Path("./devscript/moebius_minicpm_server.py"),
    Path("./moebius_minicpm_server.py"),
]

SERVER_SCRIPT = next((p for p in CANDIDATE_PATHS if p.is_file()), None)

# When triggered remotely via `kaggle kernels push` or `colab exec -f <notebook.ipynb>`,
# only the .ipynb file is sent to the remote VM. Materialize `moebius_minicpm_server.py` automatically:
EMBEDDED_SERVER_ZLIB_B64 = '...'

if SERVER_SCRIPT is None:
    SERVER_SCRIPT = Path("./moebius_minicpm_server.py")
    SERVER_SCRIPT.write_bytes(zlib.decompress(base64.b64decode(EMBEDDED_SERVER_ZLIB_B64)))
    SERVER_SCRIPT.chmod(0o755)
    print(f"📝 Materialized standalone server script -> {SERVER_SCRIPT.resolve()}")
else:
    print(f"✅ Found local server script -> {SERVER_SCRIPT.resolve()}")

In [ ]:
# @title 4. Load `.env` / Secrets & Run `moebius_minicpm_server.py` (Tailscale MagicDNS & Cloudflared)
import contextlib
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path
from urllib.request import urlopen

# --- Configurable Parameters (override via .env, Colab/Kaggle Secrets, or `papermill -p KEY VALUE`) ---
ENV_FILE = ""
TAILSCALE_AUTHKEY = ""
TAILSCALE_HOSTNAME = "moebius-gpu"
ENABLE_TAILSCALE = ""
TAILSCALE_FUNNEL = "0"
ENABLE_CLOUDFLARED = "1"
CLOUDFLARED_TOKEN = ""

# 1. Auto-load .env files if present (.env, /content/.env, Google Drive /content/drive/MyDrive/.env)
for env_candidate in [ENV_FILE, "ephemeral-inpainter-vlm.env", "/content/ephemeral-inpainter-vlm.env", ".env", "devscripts/.env", "/content/.env", "/content/drive/MyDrive/.env", "/kaggle/working/.env"]:
    if env_candidate and Path(env_candidate).is_file():
        for raw in Path(env_candidate).read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if line and not line.startswith("#") and "=" in line:
                if line.startswith("export "):
                    line = line[7:].strip()
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip("\"'"))
        print(f"🔐 Loaded environment variables from {env_candidate}")

# 2. Auto-load Google Colab Secrets (`google.colab.userdata`) or Kaggle Secrets (`kaggle_secrets`)
for secret_key in ("TAILSCALE_AUTHKEY", "TAILSCALE_HOSTNAME", "ENABLE_TAILSCALE", "ENABLE_CLOUDFLARED", "CLOUDFLARED_TOKEN"):
    if not os.environ.get(secret_key):
        with contextlib.suppress(Exception):
            from google.colab import userdata
            val = userdata.get(secret_key)
            if val:
                os.environ[secret_key] = str(val).strip()
                print(f"🔑 Loaded {secret_key} from Google Colab Secrets")
    if not os.environ.get(secret_key):
        with contextlib.suppress(Exception):
            from kaggle_secrets import UserSecretsClient
            val = UserSecretsClient().get_secret(secret_key)
            if val:
                os.environ[secret_key] = str(val).strip()
                print(f"🔑 Loaded {secret_key} from Kaggle Secrets")

# 3. Resolve final settings (Papermill cell overrides -> Env/Secrets -> Defaults)
tailscale_authkey = TAILSCALE_AUTHKEY or os.environ.get("TAILSCALE_AUTHKEY", "")
tailscale_hostname = os.environ.get("TAILSCALE_HOSTNAME", TAILSCALE_HOSTNAME or "moebius-gpu")
enable_tailscale = (ENABLE_TAILSCALE or os.environ.get("ENABLE_TAILSCALE", "")) == "1" or bool(tailscale_authkey)
tailscale_funnel = (TAILSCALE_FUNNEL or os.environ.get("TAILSCALE_FUNNEL", "0")) == "1"
cloudflared_token = CLOUDFLARED_TOKEN or os.environ.get("CLOUDFLARED_TOKEN", "")
enable_cloudflared = os.environ.get("ENABLE_CLOUDFLARED", ENABLE_CLOUDFLARED) == "1"

port = int(os.environ.get("SERVER_PORT", "8000"))
max_gpu_active = os.environ.get("MAX_GPU_ACTIVE", "2")
max_inpaint_batch = os.environ.get("MAX_INPAINT_BATCH", "4")
batch_window_ms = os.environ.get("BATCH_WINDOW_MS", "50")
max_vision_concurrency = os.environ.get("MAX_VISION_CONCURRENCY", "2")
gpu_vram_threshold = os.environ.get("GPU_VRAM_THRESHOLD", "0.85")
run_seconds = int(os.environ.get("SERVER_DURATION_SECONDS", "36000"))
smoke_test_only = os.environ.get("SMOKE_TEST_ONLY", "0") == "1"

cmd = [
    sys.executable,
    "-u",
    str(SERVER_SCRIPT),
    "--host", "0.0.0.0",
    "--port", str(port),
    "--max-gpu-active", str(max_gpu_active),
    "--max-inpaint-batch", str(max_inpaint_batch),
    "--batch-window-ms", str(batch_window_ms),
    "--max-vision-concurrency", str(max_vision_concurrency),
    "--gpu-vram-threshold", str(gpu_vram_threshold),
]
if enable_tailscale:
    cmd.extend(["--tailscale", "--tailscale-hostname", tailscale_hostname])
    if tailscale_authkey:
        cmd.extend(["--tailscale-authkey", tailscale_authkey])
    if tailscale_funnel:
        cmd.append("--tailscale-funnel")
if enable_cloudflared:
    cmd.append("--cloudflared")
    if cloudflared_token:
        cmd.extend(["--cloudflared-token", cloudflared_token])

masked_cmd = [("tskey-***" if str(x).startswith("tskey-") else x) for x in cmd]
print("🚀 Launching server:", " ".join(masked_cmd))
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
start_time = time.time()
url_regex = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")

try:
    while proc.poll() is None:
        line = proc.stdout.readline()
        if line:
            print(line, end="", flush=True)
            m = url_regex.search(line)
            if m and not public_url:
                public_url = m.group(0)
            if "Application startup complete" in line or "Uvicorn running on" in line:
                try:
                    with urlopen(f"http://127.0.0.1:{port}/api/status", timeout=5) as resp:
                        status_data = json.loads(resp.read().decode("utf-8"))
                        public_url = public_url or status_data.get("public_url")
                        print("\n" + "=" * 72)
                        print(f"✅ SERVER HEALTHY! Active Tunnel / MagicDNS URL: {public_url}")
                        print(f"   Status endpoint : {public_url or f'http://127.0.0.1:{port}'}/api/status")
                        print(f"   Inpaint endpoint: {public_url or f'http://127.0.0.1:{port}'}/api/inpaint/batch")
                        print(f"   Vision endpoint : {public_url or f'http://127.0.0.1:{port}'}/api/vision/batch")
                        print("=" * 72 + "\n", flush=True)
                except Exception as exc:
                    print(f"[Health check notice]: {exc}", flush=True)
                if smoke_test_only:
                    print("SMOKE_TEST_ONLY=1 set; stopping server cleanly after startup verification.")
                    break

        if time.time() - start_time >= run_seconds:
            print(f"⏱️ Reached SERVER_DURATION_SECONDS={run_seconds}s; shutting down server.")
            break
except KeyboardInterrupt:
    print("\n🛑 Stopping server...")
finally:
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()